In [ ]:
%%capture
import os

!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
import json

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 4096 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
#load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.



In [ ]:

"""
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-1.7B-Base", # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    #token = secret_value_0, # HF Token for gated models
)

FastLanguageModel.for_inference(base_model)
"""



In [ ]:
#FastLanguageModel.for_inference(base_model)


In [ ]:
# Use the merged 16bit GRPO model saved by the france94/qwen-grpo-v2-only-math run.
# It is mounted as a kernel-output data source (see kernel_sources in kernel-metadata.json).
# Fall back to the older HF repo if the mount is missing.
import glob
_candidates = glob.glob('/kaggle/input/**/qwen_grpo_16bit', recursive=True)
model_finetuned = _candidates[0] if _candidates else 'FrancescoArno94/Qwen3-1.7B-grpo-math-code'
print("Benchmarking model:", model_finetuned)
trained_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_finetuned, # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    #token = secret_value_0, # HF Token for gated models
)

In [ ]:
FastLanguageModel.for_inference(trained_model)


In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-3",
)

# Test inference

In [ ]:
messages = [
    {"role": "user", "content": "Continue the fibonacci sequence: 1, 1, 2, 3, 5, 8,"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    return_dict=True
).to("cuda")

In [ ]:
"""
outputs = base_model.generate(**inputs, max_new_tokens = 64, use_cache = True,
                         temperature = 0.7, min_p = 0.1)
tokenizer.batch_decode(outputs)
"""

In [ ]:
outputs = trained_model.generate(**inputs, max_new_tokens = 64, use_cache = True,
                         temperature = 0.7, min_p = 0.1)
tokenizer.batch_decode(outputs)

In [ ]:
def invoke_model(input_query, model, max_new_tokens=64, temperature=0.7, min_p=0.1):
    messages = [
        {
            "role": "user", 
             "content": input_query
        },
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True, # Must add for generation
        return_tensors = "pt",
        return_dict=True
    ).to("cuda")
    
    outputs = model.generate(**inputs, max_new_tokens = max_new_tokens, use_cache = True,
                             temperature = temperature, min_p = min_p)
    output_list = tokenizer.batch_decode(outputs)
    return output_list[0]
    
    

In [ ]:
#msg= "Resolve this equation: 2x+1=0"

#print(invoke_model(msg, base_model))

In [ ]:
msg= "Resolve this equation: 2x+1=0"

print(invoke_model(msg, trained_model))

In [ ]:
msg= "A python function that split a string"
#print(invoke_model(msg, base_model, max_new_tokens=256))
#print("-"*1000)
print(invoke_model(msg, trained_model, max_new_tokens=256))



In [ ]:
print(invoke_model(msg, trained_model, max_new_tokens=256, temperature=0.1))


In [ ]:
!pip install lm-eval

In [ ]:
from lm_eval.loggers import EvaluationTracker
tracker = EvaluationTracker(output_path="results")
#results = lm_eval.simple_evaluate(..., evaluation_tracker=tracker, log_samples=True)

In [ ]:
#from unsloth import FastLanguageModel
import lm_eval
from lm_eval.models.huggingface import HFLM

#model, tokenizer = FastLanguageModel.from_pretrained("lora_model", max_seq_length=2048, load_in_4bit=True)
#FastLanguageModel.for_inference(model)

lm = HFLM(pretrained=trained_model, tokenizer=tokenizer, batch_size=8)
res = lm_eval.simple_evaluate(model=lm, tasks=["gsm8k", "mmlu", "truthfulqa_mc2"], apply_chat_template=True,
                              evaluation_tracker=tracker)
print(res["results"])

In [ ]:
res

In [ ]:
tracker.save_results_aggregated(results=res, samples=res["samples"])


In [ ]:
for task in res["configs"]:
    tracker.save_results_samples(task_name=task, samples=res["samples"][task])

print("✅ Results saved")


In [ ]:
# 3. Save the aggregate results to a JSON file
#output_res="qwen3_sft_results.json"
#with open(output_res, "w") as f:
#    json.dump(res, f, indent=2)
    

# Optional: Display the main metrics in the notebook
#import pandas as pd
#metrics = {task: {k: v for k, v in data.items() if not k.startswith('_')} 
#           for task, data in res['results'].items()}
#display(pd.DataFrame(metrics).T)